# Milestone 1: Exploratory Data Analysis

## Detection of Hydroxychloroquine Macular Toxicity via OCT Images

Proyecto Integrador - TC5035.10

Team 45

Tecnológico de Monterrey

Sponsor: Dr. Alejandor Rodríguez García, Hospital Zambrano Hellion
Date: May 2026

Team members:

- Natalia Nevarez Tinoco – A01566204

- Alejandro Caamaño Granados – A00843392

- Diana Jimena López Nájera – A01024913

---
### Project Description
This project aims to develop a binary classification model based on Optical Coherence Tomography (OCT) images to automatically detect macular toxicity caused by long-term use of Hydroxychloroquine (HCQ), a drug used in autoinmune diseases susch as lupus and rheumatoid arthritis.

### Dataset Description
The clinical dataset was provided by Dr. Rodríguez García from Hospital Zambrano Hellion and is organized into three folders:

| Folder | Meaning | Role in model |
|--------|---------|---------------|
| `No` | No toxicity detected | Binary class: **negative** |
| `Si` | Toxicity confirmed | Binary class: **positive** |
| `Sospecha` | Suspected / inconclusive | **Excluded** from binary classification (see Section 2) |

Each folder contains OCT scan files following the naming pattern:

`AUTO{ID}_{FIRSTNAME}-{LASTNAME}_{DATE}_{TIME}_Retina_Radial_{EYE}_{aperture}_{resolution}_scan{N}.{ext}`

Where `{EYE}` is `R` (right) or `L` (left), `scan{N}` identifies the radial slice number, and `{ext}` is `jpg`, `png`, or `pdf`.

**Important:** Both image files (JPG/PNG) and PDF files are present in the folders. Inspection confirmed that **PDFs are not report wrappers** — each PDF contains exactly one OCT scan image, exported by the equipment in PDF format for certain scan slices. JPG, PNG, and PDF files for the same patient represent **different scan numbers** (e.g. scan2 as PDF, scan3 as JPG). All three formats are valid data and are loaded together.

Multiple scan files per patient are expected — the OCT equipment exports several radial cross-sectional slices per visit. Each image is treated as one data entry, and patient identity is tracked to prevent data leakage at the train/test split stage.

## 0. Google Drive Setup

In [4]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

# Automatically locate the Plaquenil folder anywhere in the team's
# Drive. No hardcoded paths; works for every teammate as long ad the folder
# structure is: Plaquenil/No, Plaquenil/Si, Plaquenil/Sospecha
search_root = '/content/drive/MyDrive'
data_root = None

found = False
for root, dirs, files in os.walk(search_root, followlinks=True):
    # Normalizamos a minúsculas para evitar errores si alguien escribió "SI" o "Si"
    dirs_low = [d.lower() for d in dirs]
    if 'no' in dirs_low and 'si' in dirs_low and 'sospecha' in dirs_low:
        data_root = Path(root)
        found = True
        break

if found:
    actual_dirs = {d.lower(): d for d in os.listdir(data_root)}
    NO_DIR = data_root / actual_dirs['no']
    SI_DIR = data_root / actual_dirs['si']
    SOSPECHA_DIR = data_root / actual_dirs['sospecha']

    print(f"Data found at: {data_root}")
    print(f"  No:       {NO_DIR}")
    print(f"  Si:       {SI_DIR}")
    print(f"  Sospecha: {SOSPECHA_DIR}")
else:
    print("ERROR: Folder 'Plaquenil' with subfolders No/Si/Sospecha not found in your Drive.")


Mounted at /content/drive
Data found at: /content/drive/MyDrive/Avance 1 - EDA/Plaquenil
  No:       /content/drive/MyDrive/Avance 1 - EDA/Plaquenil/No
  Si:       /content/drive/MyDrive/Avance 1 - EDA/Plaquenil/Si
  Sospecha: /content/drive/MyDrive/Avance 1 - EDA/Plaquenil/Sospecha


## 1. Import Libraries


In [ ]:
# pymupdf (fitz) is needed to extract images from PDF scan files.
# It is not pre-installed in Colab, so we install it once here.
!pip install pymupdf --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 30.9 MB/s eta 0:00:00


In [ ]:
import glob
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image
from pathlib import Path
import warnings

import torch
from torchvision import transforms
from torchvision.transforms import functional as F

warnings.filterwarnings('ignore')

# Visualization configuration
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

print('Libraries imported correctly')

Libraries imported correctly


In [ ]:
import shutil
from pathlib import Path

# Esta carpeta se creará
drive_dest = Path('/content/drive/My Drive/OCT_Dataset_Limpio_Final')

## 2. Dataset Loading and Structure


### 2.2 Raw File Audit

Before loading any images, we inspect every folder to count files by type and identify unique patients.

**Key findings from manual inspection of the source files:**
- Each patient has multiple files — one per OCT radial scan slice per visit (e.g. scan2, scan3 ... scan9). This is standard OCT output.
- PDF files are **not** report wrappers. Each PDF contains exactly one OCT scan image. The OCT equipment exports some scan slices as JPG/PNG and others as PDF within the same visit — they represent different scan numbers for the same patient.
- All three formats (JPG, PNG, PDF) are valid data and must all be loaded.

In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg'}
PDF_EXT = {'.pdf'}

def audit_folder(folder_path, label):
    """Count and categorize all files in a folder and report unique patients."""
    all_files = list(Path(folder_path).iterdir())
    images = [f for f in all_files if f.suffix.lower() in IMAGE_EXTS]
    pdfs   = [f for f in all_files if f.suffix.lower() in PDF_EXT]
    other  = [f for f in all_files if f.suffix.lower() not in IMAGE_EXTS | PDF_EXT]

    # Patient ID = first segment of filename (e.g. AUTO20211123110956)
    all_scan_files = images + pdfs
    patient_ids = set(f.name.split('_')[0] for f in all_scan_files)

    print(f"\n{'='*55}")
    print(f"  FOLDER : {label}")
    print(f"{'='*55}")
    print(f"  Total files          : {len(all_files)}")
    print(f"  Image files (JPG/PNG): {len(images)}")
    print(f"  PDF files (scan images): {len(pdfs)}")
    print(f"  Other files          : {len(other)}")
    print(f"  Total usable scans   : {len(images) + len(pdfs)}")
    print(f"  Unique patients      : {len(patient_ids)}")
    print(f"  Avg scans/patient    : {(len(images)+len(pdfs))/max(len(patient_ids),1):.1f}")
    return images, pdfs, patient_ids

no_images,   no_pdfs,   no_patients   = audit_folder(NO_DIR,       'No  (Normal / Negative)')
si_images,   si_pdfs,   si_patients   = audit_folder(SI_DIR,       'Si  (Toxicity / Positive)')
sosp_images, sosp_pdfs, sosp_patients = audit_folder(SOSPECHA_DIR, 'Sospecha (Suspected — excluded from binary model)')

print(f"\n>>> TOTAL usable scans (No + Si): {len(no_images)+len(no_pdfs)+len(si_images)+len(si_pdfs)}")


  FOLDER : No  (Normal / Negative)
  Total files          : 185
  Image files (JPG/PNG): 106
  PDF files (scan images): 78
  Other files          : 1
  Total usable scans   : 184
  Unique patients      : 67
  Avg scans/patient    : 2.7

  FOLDER : Si  (Toxicity / Positive)
  Total files          : 15
  Image files (JPG/PNG): 15
  PDF files (scan images): 0
  Other files          : 0
  Total usable scans   : 15
  Unique patients      : 5
  Avg scans/patient    : 3.0

  FOLDER : Sospecha (Suspected — excluded from binary model)
  Total files          : 30
  Image files (JPG/PNG): 30
  PDF files (scan images): 0
  Other files          : 0
  Total usable scans   : 30
  Unique patients      : 13
  Avg scans/patient    : 2.3

>>> TOTAL usable scans (No + Si): 199


### 2.2 PDF Scan Extraction

PDF files each contain exactly one OCT scan image. We extract them as PNG images into a temporary in memory folder (/content/pdf_extracted/) so they can be loaded identically to JPG/PNG files.

We will use `pymupdf (fitz)` to render each PDF page at 150 DPI and save it as PNG. The output filename preserves the original stem so all metadata parsing remains consistent.

In [ ]:
import fitz

EXTRACT_DIR = Path('/content/pdf_extracted')
EXTRACT_DIR.mkdir(exist_ok=True)

def extract_pdfs_to_png(pdf_list, subfolder_name):
    """
    Render each single-page PDF as a PNG and save to EXTRACT_DIR/subfolder.
    Returns a list of Path objects pointing to the extracted PNG files.
    """
    out_dir = EXTRACT_DIR / subfolder_name
    out_dir.mkdir(exist_ok=True)
    extracted = []
    errors    = []

    for pdf_path in pdf_list:
        try:
            doc = fitz.open(str(pdf_path))
            # Each OCT PDF has exactly one page
            page = doc[0]
            # Render at 150 DPI (matrix scale = 150/72 ≈ 2.08)
            mat = fitz.Matrix(150 / 72, 150 / 72)
            pix = page.get_pixmap(matrix=mat, colorspace=fitz.csGRAY)
            out_path = out_dir / (pdf_path.stem + '.png')
            pix.save(str(out_path))
            extracted.append(out_path)
            doc.close()
        except Exception as e:
            errors.append(pdf_path.name)
            print(f'  [ERROR] {pdf_path.name}: {e}')

    print(f'  Extracted {len(extracted)} PDFs → PNG  |  Errors: {len(errors)}  [{subfolder_name}]')
    return extracted

print('Extracting PDF scans...')
no_from_pdf   = extract_pdfs_to_png(no_pdfs,   'no')
si_from_pdf   = extract_pdfs_to_png(si_pdfs,   'si')
sosp_from_pdf = extract_pdfs_to_png(sosp_pdfs, 'sospecha')

# Combine native images + PDF-extracted images into one list per class
no_all_images   = no_images   + no_from_pdf
si_all_images   = si_images   + si_from_pdf
sosp_all_images = sosp_images + sosp_from_pdf

print(f'\nCombined image counts (native + extracted from PDF):')
print(f'  No       : {len(no_all_images)}  ({len(no_images)} native + {len(no_from_pdf)} from PDF)')
print(f'  Si       : {len(si_all_images)}  ({len(si_images)} native + {len(si_from_pdf)} from PDF)')

### 2.3 Duplicate Detection

We hash every image file (MD5 of raw bytes) to detect exact duplicates regardless of filename of format. This catches any files that were accidentally downloaded twice.

In [ ]:
def get_file_hash(filepath):
    """Return MD5 hash of file content."""
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def find_duplicates(image_list, label):
    """Detect duplicate images by content hash. Returns list of duplicate paths to remove."""
    hashes     = {}
    duplicates = []
    for img_path in image_list:
        h = get_file_hash(img_path)
        if h in hashes:
            duplicates.append(img_path)
            print(f'  [DUP]  {img_path.name}')
            print(f'         original: {hashes[h].name}')
        else:
            hashes[h] = img_path
    status = f'{len(duplicates)} duplicate(s) found' if duplicates else 'No duplicates found'
    print(f'{label}: {status}')
    return duplicates

print('--- Checking for duplicates ---')
no_dups   = find_duplicates(no_all_images,   'No')
si_dups   = find_duplicates(si_all_images,   'Si')
sosp_dups = find_duplicates(sosp_all_images, 'Sospecha')

# Remove duplicates from working lists
no_clean   = [f for f in no_all_images   if f not in no_dups]
si_clean   = [f for f in si_all_images   if f not in si_dups]
sosp_clean = [f for f in sosp_all_images if f not in sosp_dups]

print(f'\nFinal clean image counts after deduplication:')
print(f'  No       : {len(no_clean)}')
print(f'  Si       : {len(si_clean)}')
print(f'  Sospecha : {len(sosp_clean)}')

In [ ]:
def copy_images_to_drive(image_list, category_name):
    """Copia físicamente los archivos de la lista a una carpeta en Drive"""
    target_dir = drive_dest / category_name
    target_dir.mkdir(parents=True, exist_ok=True)

    print(f"Copiando {len(image_list)} imágenes a la carpeta: {category_name}...")

    for img_path in image_list:
        # img_path es un objeto Path de tu código anterior
        # shutil.copy2 mantiene los metadatos originales (fechas, etc)
        try:
            shutil.copy2(img_path, target_dir / img_path.name)
        except Exception as e:
            print(f"Error al copiar {img_path.name}: {e}")

# 3. Ejecutar la copia para tus tres grupos limpios
copy_images_to_drive(no_clean, 'No')
copy_images_to_drive(si_clean, 'Si')
copy_images_to_drive(sosp_clean, 'Sospecha')

print("\n¡Proceso terminado! Ya puedes revisar tu Google Drive.")

### 2.4 Filename Structure Verification

Verify the full filename pattern before parsing. Confirmed pattern:

`AUTO{ID}_{NAME}_{DATE}_{TIME}_Retina_Radial_{EYE}_{aperture}_{resolution}_scan{N}.{ext}`

We extract: patient ID, patient name, eye side (R/L), and scan number

In [ ]:
def parse_filename(filepath):
    """
    Parse full OCT filename into components.
    Handles two naming patterns:
      - Standard: AUTO{ID}_{NAME}_{DATE}_{TIME}_Retina_Radial_{EYE}_..._scan{N}.ext
      - Non-standard: {FIRSTNAME} {LASTNAME}_{DATE}_{TIME}_Retina_Radial_{EYE}_..._scan{N}.ext
    """
    stem  = Path(filepath).stem
    parts = stem.split('_')

    if stem.startswith('AUTO'):
        # Standard pattern
        patient_id   = parts[0]
        patient_name = f'{parts[1]}-{parts[2]}' if len(parts) > 2 else 'unknown'
    else:
        # Non-standard pattern — no AUTO prefix, use first segment as ID
        patient_id   = parts[0]
        patient_name = f'{parts[0]}-{parts[1]}' if len(parts) > 1 else parts[0]

    # Eye side: search for first part that is exactly 'R' or 'L'
    # (more robust than fixed index, handles spaces in filenames)
    eye_side = 'unknown'
    for p in parts:
        if p.strip() in ('R', 'L'):
            eye_side = p.strip()
            break

    scan_type = parts[-1] if parts else 'unknown'  # e.g. 'scan3'

    return patient_id, patient_name, eye_side, scan_type

# Verify on 3 samples from each folder
print('--- Filename parsing verification ---')
for folder_name, image_list in [('No', no_clean[:3]), ('Si', si_clean[:3]), ('Sospecha', sosp_clean[:3])]:
    print(f'\n{folder_name}:')
    for f in image_list:
        pid, pname, eye, scan = parse_filename(f)
        print(f'  {f.name[:70]:<70}')
        print(f'  -> id={pid} | name={pname} | eye={eye} | scan={scan}')

### 2.5 Build Master DataFrame

We load pixel level metadata for all images (No and Si only - Sospecha is kept in a seperate reference datafeame). Each row = one OCT scan image, regardless of wether it originated as JPG, PNG, or PDF.

In [ ]:
def build_records(image_list, label):
    """Extract image metadata and pixel statistics for all images in a list."""
    records = []
    for fp in image_list:
        # Skip only files that are clearly not OCT scans
        # (e.g. assignment PDFs accidentally extracted)
        if fp.stem.startswith('Avance') or fp.stem.startswith('avance'):
          print(f'  [SKIP] Not an OCT file: {fp.name[:60]}')
          continue

        patient_id, patient_name, eye_side, scan_type = parse_filename(fp)

        # Detect whether this image originated from a PDF
        original_format = 'pdf' if str(EXTRACT_DIR) in str(fp) else fp.suffix.lower()

        try:
            img = Image.open(fp).convert('L')  # grayscale
            w, h = img.size
            arr = np.array(img)
            records.append({
                'filepath'        : str(fp),
                'filename'        : fp.stem,
                'label'           : label,
                'patient_id'      : patient_id,
                'patient_name'    : patient_name,
                'eye_side'        : eye_side,
                'scan_type'       : scan_type,
                'original_format' : original_format,
                'width'           : w,
                'height'          : h,
                'aspect_ratio'    : round(w / h, 3),
                'file_size_kb'    : round(fp.stat().st_size / 1024, 1),
                'mean_intensity'  : round(arr.mean(), 2),
                'std_intensity'   : round(arr.std(), 2),
                'min_intensity'   : int(arr.min()),
                'max_intensity'   : int(arr.max()),
            })
        except Exception as e:
            print(f'  [ERROR] Could not read {fp.name}: {e}')
    return records

# Build main dataframe (binary: No + Si only)
all_records  = build_records(no_clean, 'no') + build_records(si_clean, 'si')
df           = pd.DataFrame(all_records)

# Sospecha reference dataframe (not used for modeling)
sosp_records = build_records(sosp_clean, 'sospecha')
df_sospecha  = pd.DataFrame(sosp_records)

print(f'Main dataframe (No + Si): {df.shape[0]} images, {df.shape[1]} columns')
print(f'Sospecha reference df   : {df_sospecha.shape[0]} images')
print(f'\nLabel counts: {df["label"].value_counts().to_dict()}')
df.head()

In [ ]:
 """Export the dataframes to Drive."""
# Lo guardamos en CSV para lectura humana y en Pickle para el modelo
df.to_csv(drive_dest / 'METADATA_PRINCIPAL_LIMPIO.csv', index=False)
df.to_pickle(drive_dest / 'METADATA_PRINCIPAL_LIMPIO.pkl')

# Guardamos el dataframe de sospecha
df_sospecha.to_csv(drive_dest / 'METADATA_SOSPECHA_LIMPIO.csv', index=False)
df_sospecha.to_pickle(drive_dest / 'METADATA_SOSPECHA_LIMPIO.pkl')

print(f"✅ Los DataFrames se han guardado en: {drive_dest}")
print(f"Archivos creados: METADATA_PRINCIPAL y METADATA_SOSPECHA (en formatos .csv y .pkl)")

## 3. General Dataset Structure

In [ ]:
print('=== DATASET SHAPE ===')
print(f'Rows: {df.shape[0]}    |   Columns: {df.shape[1]}')

print('\n=== DATASET TYPES ===')
print(df.dtypes)

print('\n=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found')

In [ ]:
print('=== CLASS DISTRIBUTION (imgaes) ===')
print(df['label'].value_counts())
print(f"\nClass ratio (No:Si): {df['label'].value_counts()['no']}:{df['label'].value_counts()['si']}")

print('\n=== PATIENTS PER CLASS ===')
print(df.groupby('label')['patient_id'].nunique().rename('unique_patients'))

print('\n === IMAGES PER PATIENT ===')
per_patient = (
    df.groupby(['label', 'patient_id', 'patient_name'])['filename']
    .count()
    .reset_index()
    .rename(columns={'filename': 'num_images'})
    .sort_values(['label', 'num_images'], ascending=[True, False])
)
print(per_patient.to_string(index=False))

print('\n=== FILE FORMAT BREAKDOWN')
print(df.groupby(['label', 'original_format']).size().reset_index(name='count').to_string(index=False))

print('\n=== EYE SIDE BREAKDOWN')
print(df.groupby(['label', 'eye_side']).size().reset_index(name='count').to_string(index=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set a clean, professional theme
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'sans-serif'

# Define counts and pat_counts
counts = df['label'].value_counts()
pat_counts = df.groupby('label')['patient_id'].nunique()

# Create the figure
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Reshape data for a grouped comparison (Tidy Data format)
data = {
    'Category': ['Healthy (No)', 'Toxicity (Yes)', 'Healthy (No)', 'Toxicity (Yes)'] and pat_counts.get('no', 0) and pat_counts.get('si', 0),
    'Count': [counts['no'], counts['si'], pat_counts['no'], pat_counts['si']],
    'Metric': ['Total Images', 'Total Images', 'Unique Patients', 'Unique Patients']
}
plot_df = pd.DataFrame(data)

# Grouped Bar Plot
splot = sns.barplot(
    data=plot_df,
    x='Metric', y='Count', hue='Category',
    palette=['#2c3e50', '#e74c3c'],  # Professional Navy and Clinical Red
    edgecolor='black', linewidth=0.5, alpha=0.9
)

# --- AESTHETIC ENHANCEMENTS ---

# 1. Data labels above bars
for p in splot.patches:
    if p.get_height() > 0:
        splot.annotate(format(p.get_height(), '.0f'),
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha = 'center', va = 'center',
                       xytext = (0, 9),
                       textcoords = 'offset points',
                       fontsize=12, fontweight='bold', color='#34495e')

# 2. Titles and labels
plt.title('Dataset Distribution Analysis: Images vs. Unique Patients', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('') # Removed redundant "Metric" label
plt.ylabel('Total Count', fontsize=12)

# 3. Despine (removes top and right borders)
sns.despine()

# 4. Legend positioning
plt.legend(title='Medical Classification', title_fontsize='13', fontsize='11', loc='upper right')

# 5. Insight footnote
plt.figtext(0.5, -0.05,
            f"Note: Image imbalance ratio is {counts['no']/counts['si']:.1f}:1. "
            "Severe class imbalance detected; requires data augmentation or class-weighting strategies.",
            ha="center", fontsize=10, style='italic', color='#7f8c8d')

plt.tight_layout()
plt.show()

In [ ]:
print('=== DESCRIPTIVE STATISTICS (all numerical features) ===')
df[['width', 'height', 'aspect_ratio', 'file_size_kb', 'mean_intensity', 'std_intensity', 'max_intensity', 'min_intensity']].describe()

In [ ]:
print('=== DESCRIPTIVE STATISTICS BY CLASS ===')
df.groupby('label')[['width', 'height', 'aspect_ratio', 'file_size_kb', 'mean_intensity', 'std_intensity']].describe().round(2)

## 4. Univariate Analysis

### 4.1 Distribution of Pixel Intensity

Mean and standard deviation of pixel intensity are the primary numerical features derived from each image. Higher mean intensity indicates brighter retinal reflectivity, while the distribution shape reveals whether the classes overlap or are seperable by intensity alone.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, title, color in zip(
    axes,
    ['mean_intensity', 'std_intensity'],
    ['Mean Pixel Intensity', 'Std Deviation of Pixel Intensity'],
    ['#3498db', '#9b59b6']
):
    ax.hist(df[col], bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(df[col].mean(), color='#e74c3c', linestyle='--',
               linewidth=1.5, label=f'Mean = {df[col].mean():.1f}')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Distribution of Pixel Intensity Metrics (All Classes)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_intensity_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

# 1. Define the core metrics that actually impact AI training
# We exclude min/max intensity as they are almost always 0 and 255
core_metrics = ['width', 'height', 'mean_intensity', 'std_intensity', 'file_size_kb']

# 2. Calculate the average values per class
stats_summary = df.groupby('label')[core_metrics].mean()

# 3. Calculate the percentage difference between 'si' (Toxicity) and 'no' (Healthy)
# Formula: ((Si - No) / No) * 100
diff_pct = ((stats_summary.loc['si'] - stats_summary.loc['no']) / stats_summary.loc['no'] * 100).to_frame().T
diff_pct.index = ['Difference (%)']

# 4. Combine into a single master table
final_stats_table = pd.concat([stats_summary, diff_pct])

# 5. Apply professional styling (Round to 2 decimals and highlight significant differences)
def highlight_imbalance(val):
    """Highlights differences greater than 15% in red to flag potential bias."""
    if isinstance(val, float) and (val > 15 or val < -15):
        return 'color: #e74c3c; font-weight: bold'
    return ''

print("=== STRATEGIC DATASET SUMMARY (AVERAGES) ===")
styled_table = final_stats_table.style.format("{:.2f}").applymap(
    highlight_imbalance,
    subset=pd.IndexSlice[['Difference (%)'], :]
)

# Display the table
styled_table


### 4.2 Distribution of Image Dimensions and File Size

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = {'no': '#2c3e50', 'si': '#e74c3c'} # Navy Blue for Healthy, Clinical Red for Toxicity

metrics = ['width', 'height', 'file_size_kb']
titles = ['Image Width (px)', 'Image Height (px)', 'File Size (KB)']

# 3. Generate the plots
for i, metric in enumerate(metrics):
    sns.histplot(
        data=df,
        x=metric,
        hue='label',
        multiple="layer",      # Overlays the bars
        palette=colors,
        ax=axes[i],
        alpha=0.7,             # Transparency to see overlapping areas
        kde=False,             # This removes the line
        edgecolor='white',     # Better separation between bars
        linewidth=1
    )

    # Title and Label styling
    axes[i].set_title(titles[i], fontweight='bold', fontsize=14, pad=15)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency', fontsize=11)

    # Modern finish: remove top and right borders
    sns.despine()

# 4. Global Figure Title
plt.suptitle('Structural Distribution Analysis: Healthy vs. Toxicity',
             fontsize=20, fontweight='bold', y=1.05)

plt.tight_layout()
plt.show()

### 4.3 Sample Images per Class

Visual inspection of OCT images from each class to confirm data quality and understand what the model will be learning from.

In [ ]:
def show_sample_images(df, label, color, n=6):
    sample_paths = df[df['label'] == label]['filepath'].tolist()[:n]
    n_actual = len(sample_paths)
    if n_actual == 0:
        print(f'No images found for label: {label}')
        return
    fig, axes = plt.subplots(1, n_actual, figsize=(3 * n_actual, 3))
    if n_actual == 1:
        axes = [axes]
    for i, (ax, fp) in enumerate(zip(axes, sample_paths)):
        img = Image.open(fp).convert('L')
        ax.imshow(img, cmap='gray')
        ax.set_title(f'Scan {i+1}', fontsize=8)
        ax.axis('off')
    axes[0].set_ylabel(label.upper(), fontsize=11, fontweight='bold', color=color)
    plt.suptitle(f'Sample OCT Images — Class: {label.upper()}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'plot_mosaic_{label}.png', bbox_inches='tight')
    plt.show()

show_sample_images(df, 'no', '#2ecc71')
show_sample_images(df, 'si', '#e74c3c')

## 5. Bivariate / Multivariate Analysis

### 5.1 Correlation Matrix: Numerical Values

In [ ]:
# width, height, and aspect_ratio are excluded — all images share
# the same resolution (860x580), so these features have zero variance
# and produce artifically perfect correlations
numeric_cols = ['file_size_kb', 'mean_intensity', 'std_intensity',
                'min_intensity', 'max_intensity']
corr = df[numeric_cols].corr()

plt.figure(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.5)
plt.title('Correlation Matrix — Pixel Intensity Features', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_correlation.png', bbox_inches='tight')
plt.show()

### 5.2 Pixel Intensity by Class - Boxplot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, title in zip(
    axes,
    ['mean_intensity', 'std_intensity'],
    ['Mean Pixel Intensity by Class', 'Std Deviation by Class']
):
    df.boxplot(column=col, by='label', ax=ax,
               boxprops=dict(color='#2c3e50'),
               medianprops=dict(color='#e74c3c', linewidth=2))
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel(col)

plt.suptitle('Pixel Intensity Comparison Between Classes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_boxplot_intensity.png', bbox_inches='tight')
plt.show()

### 5.3 Pixel Histogram - Sample Images per Class

In [ ]:
no_sample_path = df[df['label'] == 'no'].iloc[0]['filepath']
si_sample_path = df[df['label'] == 'si'].iloc[0]['filepath']

img_no = np.array(Image.open(no_sample_path).convert('L'))
img_si = np.array(Image.open(si_sample_path).convert('L'))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].imshow(img_no, cmap='gray')
axes[0, 0].set_title('OCT — No Toxicity (Normal)', fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_si, cmap='gray')
axes[0, 1].set_title('OCT — Si Toxicity', fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].hist(img_no.flatten(), bins=64, color='#2ecc71', alpha=0.8)
axes[1, 0].set_title('Pixel Histogram — No (Normal)', fontweight='bold')
axes[1, 0].set_xlabel('Intensity (0–255)')
axes[1, 0].set_ylabel('Frequency')

axes[1, 1].hist(img_si.flatten(), bins=64, color='#e74c3c', alpha=0.8)
axes[1, 1].set_title('Pixel Histogram — Si (Toxicity)', fontweight='bold')
axes[1, 1].set_xlabel('Intensity (0–255)')
axes[1, 1].set_ylabel('Frequency')

plt.suptitle('Visual Comparison and Pixel Histograms by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_pixel_histograms.png', bbox_inches='tight')
plt.show()

### 5.4 Numerical Features by Class - Pairplot

In [ ]:
pairplot_cols = ['mean_intensity', 'std_intensity', 'file_size_kb', 'aspect_ratio', 'label']
palette = {'no': '#2ecc71', 'si': '#e74c3c'}

g = sns.pairplot(df[pairplot_cols], hue='label', palette=palette,
                 plot_kws={'alpha': 0.6}, diag_kind='kde')
g.fig.suptitle('Pairplot of Numerical Features by Class', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('plot_pairplot.png', bbox_inches='tight')
plt.show()

### 5.5 Scans per Patient by Class

In [ ]:
per_patient = (
    df.groupby(['label', 'patient_id', 'patient_name'])['filename']
    .count()
    .reset_index()
    .rename(columns={'filename': 'num_images'})
)

colors = {'no': '#2ecc71', 'si': '#e74c3c'}

for label in ['no', 'si']:
    subset = per_patient[per_patient['label'] == label].sort_values('num_images', ascending=True)
    # Dynamic height based on number of patients
    fig_height = max(6, len(subset) * 0.35)
    fig, ax = plt.subplots(figsize=(9, fig_height))
    ax.barh(subset['patient_name'], subset['num_images'],
            color=colors[label], alpha=0.8, edgecolor='white')
    ax.set_title(f'Scans per Patient — {label.upper()}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Number of Images')
    ax.set_ylabel('Patient')
    ax.tick_params(axis='y', labelsize=8)
    plt.tight_layout()
    plt.savefig(f'plot_scans_per_patient_{label}.png', bbox_inches='tight')
    plt.show()

In [ ]:
img_counts = df.groupby(['label', 'patient_id']).size().reset_index(name='scans_per_patient')


plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")


sns.boxplot(data=img_counts, x='label', y='scans_per_patient', palette=['#2c3e50', '#e74c3c'], width=0.5)
sns.stripplot(data=img_counts, x='label', y='scans_per_patient', color='black', alpha=0.3, jitter=True)

plt.title('Scan Distribution per Patient (Data Equity)', fontsize=16, fontweight='bold')
plt.xlabel('Clinical Class', fontsize=12)
plt.ylabel('Number of Scans', fontsize=12)
sns.despine()

plt.tight_layout()
plt.show()

### 5.6 Average Image per Class

To detect systemic visual differences between the two classes at a population level, we compute the pixel wise average of all images within each class. All images are first resampled to a common resolution (860x580 px, the most common native resolution) before averaging. The third panel shows the difference between the two averages, highlighting regions where the classes diverge most strongly.

If a coherent retinal structure is visible in the average image, it suggests that the macula is consistently positioned across scans. If the difference map shows distinct bright or dark regions, it indicates systematic intensity differences between toxic and non toxic scans that the model can potentially learn from.

In [ ]:
from PIL import Image
import numpy as np

# Target size to resample all images to a common resolution before averaging
TARGET_SIZE = (860, 580) # most common dimension in the dataset

def compute_average_image(df, label, target_size=TARGET_SIZE):
  '''Compute the pixel wise average of all images in a given class'''
  paths = df[df['label'] == label]['filepath'].tolist()
  accumulator = np.zeros((target_size[1], target_size[0]), dtype=np.float64)
  count = 0
  for fp in paths:
    try:
      img = Image.open(fp).convert('L').resize(target_size)
      accumulator += np.array(img, dtype=np.float64)
      count += 1
    except Exception as e:
      print(f'  [SKIP] {fp}: {e}')
  return (accumulator / count).astype(np.uint8), count

print('Computing average images per class...')
avg_no, n_no = compute_average_image(df, 'no')
avg_si, n_si = compute_average_image(df, 'si')

# Compute the difference (Si average minus No average)
# Highlights regions where the classes systematically differ
diff = avg_si.astype(np.int16) - avg_no.astype(np.int16)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(avg_no, cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'Average OCT - No Toxicity\n(n = {n_no} images)', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(avg_si, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'Average OCT - Si Toxicity\n(n = {n_si} images)', fontweight='bold')
axes[1].axis('off')

# Difference image: red = brighter in Si, blue = brighter in No
im = axes[2].imshow(diff, cmap='RdBu_r', vmin=-30, vmax=30)
axes[2].set_title('Difference (Si - No)\nred = brighter in Si, blue = brighter in No',
                  fontweight='bold')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle('Average OCT Images per Class - Population Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_average_image_per_class.png', bbox_inches='tight')
plt.show()

### 5.7 Pixel Intensity Heatmaps and Variable Maps per Class

To compliment the grayscale averages from Section 5.6, we visualize the same data as heatmaps using a perceptual color scale, which makes intensity differences easier to interpret. We also compute pixel wise standard deviation maps for each class, which show where the scans vary most across patients.

The mean heatmaps reveal where the retinal signal concentrates on average for each class. The standard deviation maps reveal which image regions are consistent across scans and which vary heavily, high variance can indicate inconsistent patient positioning, different scan angles, or biological variability between patients.

In [ ]:
def compute_class_statistics(df, label, target_size=TARGET_SIZE):
  '''Compute pixel wise mean and standard deviation for a class'''
  paths = df[df['label'] == label]['filepath'].tolist()
  stack = []
  for fp in paths:
    try:
      img = Image.open(fp).convert('L').resize(target_size)
      stack.append(np.array(img, dtype=np.float64))
    except Exception as e:
      print(f'   [SKIP] {fp}: {e}')
  stack = np.array(stack)
  mean_img = stack.mean(axis=0)
  std_img = stack.std(axis=0)
  return mean_img, std_img, len(stack)

print('Computing pixel wise statistics per class...')
mean_no, std_no, n_no = compute_class_statistics(df, 'no')
mean_si, std_si, n_si = compute_class_statistics(df, 'si')

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# Row 1: mean intensity heatmaps
im00 = axes[0, 0].imshow(mean_no, cmap='hot', vmin=0, vmax=120)
axes[0, 0].set_title(f'Mean Intensity — No Toxicity (n = {n_no})',
                     fontweight='bold')
axes[0, 0].axis('off')
plt.colorbar(im00, ax=axes[0, 0], fraction=0.046, pad=0.04, label='Pixel intensity')

im01 = axes[0, 1].imshow(mean_si, cmap='hot', vmin=0, vmax=120)
axes[0, 1].set_title(f'Mean Intensity — Si Toxicity (n = {n_si})',
                     fontweight='bold')
axes[0, 1].axis('off')
plt.colorbar(im01, ax=axes[0, 1], fraction=0.046, pad=0.04, label='Pixel intensity')

# Row 2: standard deviation maps
im10 = axes[1, 0].imshow(std_no, cmap='viridis', vmin=0, vmax=60)
axes[1, 0].set_title(f'Std Deviation — No Toxicity\n(variability across scans)',
                     fontweight='bold')
axes[1, 0].axis('off')
plt.colorbar(im10, ax=axes[1, 0], fraction=0.046, pad=0.04, label='Std deviation')

im11 = axes[1, 1].imshow(std_si, cmap='viridis', vmin=0, vmax=60)
axes[1, 1].set_title(f'Std Deviation — Si Toxicity\n(variability across scans)',
                     fontweight='bold')
axes[1, 1].axis('off')
plt.colorbar(im11, ax=axes[1, 1], fraction=0.046, pad=0.04, label='Std deviation')

plt.suptitle('Pixel Intensity Heatmaps and Variability Maps per Class',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_intensity_heatmap_per_class.png', bbox_inches='tight')
plt.show()

## 6. Preprocessing

### 6.1 Decisions Made During Raw Data Audit

| Issue | Decision | Justification |
|-------|----------|---------------|
| PDF files in folders | **Included** — extracted via `pymupdf` | PDFs contain one OCT scan image each (confirmed by inspection). The OCT equipment exports some slices as PDF and others as JPG/PNG. Excluding them would discard valid data. |
| `Sospecha` folder | Excluded from binary df | Inconclusive labels cannot be reliably used for supervised binary classification. Kept as a reference dataframe. |
| Duplicate files | Removed (if any found) | Exact content duplicates detected via MD5 hash to avoid inflating training data. |
| Multiple scans per patient | Kept | Each radial scan slice is a valid independent image. Patient ID is tracked to enforce patient-level train/test splitting. |
| Right and left eye scans | Kept (both) | Both eyes may show toxicity signs. Eye side (`R`/`L`) is recorded as a metadata feature. |

### 6.2 Class Imbalance Assessment

In [ ]:
counts = df['label'].value_counts()
total = counts.sum()

no_count = counts.get('no', 0)
si_count = counts.get('si', 0)
ratio = no_count / si_count if si_count > 0 else 0


fig, ax = plt.subplots(figsize=(7, 7))
colors = ['#2c3e50', '#e74c3c']


wedges, texts, autotexts = ax.pie(
    [no_count, si_count],
    labels=['Healthy (No)', 'Toxicity (Si)'],
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    pctdistance=0.85, # Moves the percentage labels further out
    wedgeprops={'width': 0.3, 'edgecolor': 'white', 'linewidth': 3} # Creates the "Donut" hole
)


plt.setp(autotexts, size=12, weight="bold", color="white")
plt.setp(texts, size=12, weight="bold")

ax.text(0, 0, f'TOTAL\n{total}', ha='center', va='center', fontsize=20, fontweight='bold', color='#34495e')


ax.set_title('Clinical Class Proportion', fontsize=18, fontweight='bold', pad=20)


plt.text(0, -1.2, f"Imbalance Ratio: {ratio:.1f}:1", ha='center', fontsize=11, style='italic', color='#7f8c8d')

plt.tight_layout()
plt.savefig('plot_class_proportion_modern.png', bbox_inches='tight', dpi=300)
plt.show()

print(f"\n" + "="*30)
print(f"CLASS IMBALANCE REPORT")
print(f"="*30)
print(f"Total Samples: {total}")
print(f"Healthy (No):  {no_count}")
print(f"Toxicity (Si): {si_count}")
print(f"Ratio (No:Si): {ratio:.1f}:1")
print("-" * 30)
print("ACTION PLAN FOR MILESTONE 2:")
print("  • Apply Class-Weighted Loss to penalize minority misclassification.")
print("  • Execute Synthetic Data Augmentation (Oversampling) for 'Si' class.")
print("  • Metrics focus: Prioritize PR-Curve and F1-Score over Accuracy.")

### 6.3 Image Dimension Standardization Check

Deep learning models require unirform input dimensions. We identify whether all images share the same resolution or resizing will be needed.

In [ ]:
dim_counts = df.groupby(['width', 'height']).size().reset_index(name='count').sort_values('count', ascending=False)
print('Image dimension frequency:')
print(dim_counts.to_string(index=False))

n_unique_dims = len(dim_counts)
if n_unique_dims > 1:
    most_common = dim_counts.iloc[0]
    print(f'\n{n_unique_dims} distinct dimension(s) found.')
    print(f'Most common: {int(most_common["width"])} x {int(most_common["height"])} px ({most_common["count"]} images)')
    print('Recommendation: Resize all images to a uniform size (e.g. 224x224) before training.')
else:
    print('\nAll images share the same dimensions. No resizing required for consistency.')

### 6.4 Pixel Intensity Normalization Plan

Pixel values range from 0-255. We varify the actual range in this dataset and document the normalization strategy.

In [ ]:
print('=== PIXEL INTENSITY RANGE ===')
print(f"Global min intensity  : {df['min_intensity'].min()}")
print(f"Global max intensity  : {df['max_intensity'].max()}")
print(f"Mean of mean_intensity: {df['mean_intensity'].mean():.2f}")
print(f"Std  of mean_intensity: {df['mean_intensity'].std():.2f}")

print("""
Preprocessing plan for model training (to be implemented in Milestone 2):
  1. All images already in grayscale (confirmed during metadata extraction)
  2. Resize to uniform target size, most likely 224x224 (compatible with VGG, ResNet, EfficientNet)
  3. Normalize pixel values: divide by 255.0 to scale to [0.0, 1.0]
  4. Optional: apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
     to enhance retinal layer visibility, particularly useful for low-contrast scans
""")

## 7. Possible Data Augmentation methods

As shown previously, our dataset presents two major problems that challenge the accuracy of our binary classification model:

1.   The class imbalance of the dataset overrepresenting “No toxicity”, which would lead to bias toward the majority class.
2.   The low number of data points in the class “Toxicity confirmed”, which would lead to poor model generalization.

Although the first problem could technically be solved by undersampling the majority class, this also would lead to poor generalization by further decreasing the size of the dataset. To remedy both problems, we think **data augmentation of the “Toxicity confirmed” class** is the best solution. This is a very popular technique in computer vision and, specifically, in medical imaging, where large datasets aren’t typically available due to not enough patients for some diseases, patients do not want to allow their images to be used, lack of medical equipment, inability to obtain images that meet the desired criteria, etc [1].

There are many methods to do data augmentation (DA). Goceri (2023) made a comprehensive review of the most common DA techniques, and argues that these should be chosen carefully according to image types. We’ll analyze these methods, from basic techniques (e.g. geometric transformations, cropping, etc) to novel Deep learning techniques (Generative Adversarial Networks and Variational Autoencoders) to determine which are the most suitable for our images. It’s very probable that a combination of these methods will give us the best results.

### 7.1 Flipping horizontally

This consists of flipping the image left-right, creating a synthetic, mirrored image of the original. Since our dataset contains left and right scans of each patient, flipping the left eye scan will generate a synthetic right eye scan, and flipping the right eye scan will generate a synthetic left eye scan. This method is commonly used for augmentations of other medical images, such as CT lung images and eye fundus images [1].

Flipping horizontally will easily allow us to double the number of usable data, consuming little time and computing power.

In [ ]:
with Image.open(no_sample_path).convert('L') as img_no_dataaug:
    # RandomHorizontalFlip(p=0.5) will flip half the time during training
    flip_transform = transforms.RandomHorizontalFlip(p=1.0)
    img_no_dataaugd = flip_transform(img_no_dataaug)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(img_no_dataaug, cmap='gray')
    axes[0].set_title('Original image', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_no_dataaugd, cmap='gray')
    axes[1].set_title('Augmented image', fontweight='bold')
    axes[1].axis('off')

    plt.suptitle('Visual Comparison of Horizontal flip', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plot_dataaug_hflip.png', bbox_inches='tight')
    plt.show()

### 7.2 Translation and random cropping

Translation consists of shifting images left, right, up, or down, avoiding positional bias in the data. As the original image is translated in a direction, the remaining space is filled with a constant value (usually 0s) or random noise. This padding preserves the spatial dimensions of the image post-augmentation. Random cropping can also be used to provide an effect very similar to translations.

These methods have been used for augmentations of other medical images, such as brain MR images, CT lung images, and eye fundus images [1]. Translation and random cropping, both geometric transformations, must be manually observed to make sure they do not remove critical features of the image, altering its label [3].

Since our dataset’s macular images are not always centered, these methods, used with moderation to not remove important image features, will allow us to lower the model’s positional bias while also generating more minority class images.

In [ ]:
with Image.open(no_sample_path).convert('L') as img_no_dataaug:
    # (0.3, 0.3) means the image can move up to 30% in any direction
    translate_transform = transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))
    img_no_dataaugd = translate_transform(img_no_dataaug)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(img_no_dataaug, cmap='gray')
    axes[0].set_title('Original image', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_no_dataaugd, cmap='gray')
    axes[1].set_title('Augmented image', fontweight='bold')
    axes[1].axis('off')

    plt.suptitle('Visual Comparison of Random translation', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plot_dataaug_rtranslation.png', bbox_inches='tight')
    plt.show()

### 7.3 Rotation

Rotation augmentations are done by rotating the image right or left on an axis between 1° and 359°. The safety of rotation augmentations is heavily determined by the rotation degree parameter [3]. Rotation is commonly used for augmentations of other medical images, such as brain MR images, CT lung images, breast mammography images, and eye fundus images [1].

Since maculae often present with some rotation in OCT scans, we believe that adding random, small rotations to the data augmentation process will generate realistic variations that will benefit the model’s robustness.

In [ ]:
with Image.open(no_sample_path).convert('L') as img_no_dataaug:
    rotate_transform = transforms.RandomAffine(degrees=15)
    img_no_dataaugd = rotate_transform(img_no_dataaug)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(img_no_dataaug, cmap='gray')
    axes[0].set_title('Original image', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_no_dataaugd, cmap='gray')
    axes[1].set_title('Augmented image', fontweight='bold')
    axes[1].axis('off')

    plt.suptitle('Visual Comparison of Random rotation', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plot_dataaug_rrotation.png', bbox_inches='tight')
    plt.show()

### 7.4 Noise injection

Noise injection consists of injecting a matrix of random values usually drawn from a Gaussian distribution. Adding noise to images can help CNNs learn more robust features [3].

Of the eleven DA methods Goceri (2023) compared, noise injection showed one of the best classification accuracies with CT lung, breast mammography, and eye fundus images. Since these images share properties with our dataset (grayscale images, imaging of paired organs enabling horizontal flipping), we believe this method will also give us good results.

In [ ]:
with Image.open(no_sample_path).convert('L') as img_no_dataaug:
    # Convert PIL image to Tensor for noise injection
    img_tensor = F.to_tensor(img_no_dataaug)
    noise_level = 0.05
    noisy_tensor = img_tensor + torch.randn_like(img_tensor) * noise_level

    # Clamp and convert back to PIL for showing
    img_no_dataaugd = F.to_pil_image(torch.clamp(noisy_tensor, 0, 1))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(img_no_dataaug, cmap='gray')
    axes[0].set_title('Original image', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_no_dataaugd, cmap='gray')
    axes[1].set_title('Augmented image', fontweight='bold')
    axes[1].axis('off')

    plt.suptitle('Visual Comparison of Random noise injection', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plot_dataaug_rnoiseinj.png', bbox_inches='tight')
    plt.show()

### 7.5 GANs

Generative Adversarial Networks are class of deep generative models composed of two separate networks: a generator and a discriminator. During the training process, the generator (typically a reversed CNN) is trained to replicate data points so that the discriminator (typically a CNN) cannot distinguish between real data points and the generated data points. These two networks are trained simultaneously in an adversarial manner, ideally until both the generator and discriminator are equally effective at their tasks. Once trained, new data points can be synthesized [2]. GANs present a novel approach to DA.

The main challenging issues of GAN based image augmentations are generating images with high quality (i.e., high-resolution, clear) and maintaining training stability. To overcome those issues, several GAN variants have been developed [1].

### 7.6 VAEs

Variational autoencoders are generative models used to generate new data in the form of variations of the input data. They’re composed of an encoder that learns to isolate the important latent variables from training data, and a decoder that then uses those latent variables to reconstruct the input data. Since VAEs encode a continuous, probabilistic representation of the latent space, they can use variational inference to generate new data samples that resemble the original input data [4].

## 8. EDA Conclusions / Findings

### 8.1 Dataset Composition and Structure



The final datset used for binary classification contains **198 images** from **71 unique patients** across two classes: `no` (no toxicity, 183 images, 69 patients) and `si` (toxicity confirmed, 15 images, 5 patients). A third folder, `Sospecha` (suspected/inconclusive, 30 images, 13 patients), was kept as a reference dataframe but excluded from the binary model - its labels are clinically inconclusive and cannot be reliably used for supervised learning.

Each patient contributes multiple scan images, one per radial OCT slice per clinical visit. This is standard equipment output and each image as an independent data entry. Patient identity is tracked via `patient_id` to ensure that when the train/test split is performed in the next milestone, it is done **at the patient level** - all scans from a given patient go exclusively to either training or testing, never both. This prevents data leakage, where the model could exploit the visual similarity between scans of the same eye rather than laerning generalizable toxicity patterns.

### 8.2 File Format Discovery

A key finding during the audit was that **PDF files in the dataset are not report wrappers**; each PDF contains exactly one OCT scan image, exported by the equipment in PDF format for certain scan slices. For the same patient visit, some scan numbers are saved as JPG or PNG and others as PDF. All three formats represent valid, distinct scan images. PDFs were extracted to PNG using `pymupdf` at 150 DPI before processing. Excluding PDFs would have discarded 71 valid scans from the No class alone.

### 8.3 Data Quality Issues Encountered and Resolved

Several data quality issues were identified and resolved during loading:

- **Duplicate files:** One file in the No folder (`scan5 (1).png`) was an exact content duplicate of `scan5.png`, detected via MD5 hash comparison. It was removed, reducing the No count from 184 to 183 native images.
- **Non-standard filenames**: Six files in the NO folder did not follow the expected `AUTO{ID}_...` naming convention; their filenames begin directly with the patient's first name (e.g. `ISABEL MARGARITA_CORTES CONTRERAS_...`, `KARLA ALEJANDRA_MOTA MARTINEZ_...`, `SARA MARIA_VALDEZ PEREZ_...`). These were initially skipped by the `AUTO` prefix filter and only recovered after updating `parse_filename` to handle both naming patterns. These files are real OCT scans and their omission would have been a silent data loss. Dr. Rodriguez Garcia should be informed of this inconsistency in the equipment export settings.
- **Patient appearing in both classes**: Patient appearing in both classes: Patient `AUTO20240902130109 MARIA ALEJANDRA-GARZA LOZANO` has 2 images in the No class and 2 images in the Si class. This is clinically plausible — she may have had a normal scan at an earlier visit and developed toxicity by a later visit. Both sets of scans are kept with their original labels since they reflect her true clinical state at each visit. However, for the train/test split, all 4 of her scans must be assigned to the same set (either entirely
to training or entirely to testing) to prevent data leakage from the
model recognizing her specific retinal features.

### 8.4 Real World Data Collection Challenges

This dataset reflect the realities of working with clinical data in a real hospital setting, which differs significantly from the clean, pre-packaged datasets usually used in academic machine learning classes.

The data was shared by Dr. Rodríguez García via a OneDrive link containing the three class folders. Beacuse there was no automated access to the shared drive, **each file had to be downloaded manually**, one by one, and then re uploaded to the Google Drive for use in this notebook. This manual process introduced several human errors that had to be detected and corrected programmatically:

- **Accidental duplicate download**: One scan (`scan5.png` for patient MIGUEL-ANGEL HERNANDEZ EGREMY) was downloaded twice, resulting in a file named `scan5 (1).png` alongside the original. Without the MD5 hash duplicate detection implemented in Section 2, this duplicate would have silently inflated the training data and slightly biased the model toward the patient's retinal features.

- **Miscount between OneDrive and local folder**: OneDrive reported 183 selected fles in the No folder, but the local Finder showed 186 items, a discrepancy explained by the presence of PDFs, duplicates, and a rogue rubric PDF (that has now been removed) that had been accidentally placed in the folder.

- **Inconsistent file formats**: Some scans were available only as PDF, others only as JPG or PNG, with on clear pattern. This required building a PDF extraction pipeline that would not have been necessary with a standardized datase.

These challenges are not unique to this project, they are representative of what data scientists face when working directly with hospital systems that were designed for clinical documentation, not machine learning. OCT equipment exports data format its firmware is configured for, naming conventions are not always enforced, and files accumulate in shared drives without strict organization. Building robust, defensive data loading code, as done in Section 2, is therefore a necessary part of any real world medical AI project.


### 8.5 Image Characteristics

All 198 images are grayscale OCT cross-sectional scans of the retina with consistent scale bars (200 µm) and directional orientation markers. Two resolution variants exist in the dataset: **860×580 px** (native JPG/PNG exports) and **1755×1240 px** (PDF-extracted images, rendered at 150 DPI). Since deep learning models require uniform input dimensions, all images will be resized to a standard size (most likely **224×224 px**) before training. The aspect ratio is preserved at approximately 1.48:1 across both resolution variants.

Every image has `min_intensity = 0` and `max_intensity = 255`, confirming full dynamic range usage. The large background areas in OCT images always drive the minimum to zero, while bright retinal layer reflections always reach 255. These two features carry no discriminative information and were excluded from the correlation analysis.

### 8.6 Pixel Intensity Analysis

Mean pixel intensity across the full dataset ranges from 31.98 to 80.00, with an overall mean of 53.7. The majority of each image is dark background, which keeps mean values low. The correlation matrix revealed that `mean_intensity`, `std_intensity` and `file_size_kb` are correlated (r ≥ 0.90), meaning they all capture the same underlying property, overall image brightness. Brighter images have more pixel variation and compress less efficiently, resulting in larger file sizes. For the model, these metadata features are redundant, the network will learn directly from raw pixel values.

The boxplots and pixel histograms comparing No and Si classes show whether the two classes are separable by intensity alone. If the distributions overlap significantly, it confirms that a deep learning model is necesarry to detect subtle structural differences in retinal layer morphology that simple intensity statistics cannot capture.

### 8.7 Class Imbalance

The dataset is severely imbalanced: **183 No images vs. 15 Si images**, a ratio of approximately **12:1**. This is clinically realistic, HCQ toxicity is rare among patients taking the drug. However, training a model on this raw distribution would result in a classifier that predicts "no toxicity" for every patient and still achieves ~92% accuracy, which is clinically useless.

Section 7 outlines six data augmentation methods identified as candidates
for addressing the severe class imbalance and small dataset size. The team
will begin with simpler geometric transformations (horizontal flipping,
rotation, translation, noise injection) supported by existing literature
for similar medical imaging tasks, and evaluate whether generative
approaches (GANs, VAEs) are needed if initial results are insufficient.

### 8.8 Future Use of Sospecha

The Sospecha class was excluded from training and evaluation because its labels are inconclusive. However, once the model is trained, running it on the Sospecha cases represents a clinically interesting experiment: if the model consistently predicts toxicity for suspected cases, that output could serve as a second opinion tool to support Dr. Rodríguez García's clinical decision-making for ambiguous patients.

### 8.9 Clinical Consultation on Scan Angle Metadata

A consultation with Dr. Rodríguez García clarified two points about
the radial scan structure. First, the angle (meridian) of each scan
is not standardized — it varies by equipment, software, and even
across visits for the same patient. Second, the angle is not
clinically relevant for detecting HCQ toxicity; the 360° radial
sampling is performed to fully survey the macula for other
pathologies (e.g., subretinal or intraretinal fluid). For this reason,
the scan angle was not extracted as a feature. Each of the radial
slices is treated as an independent valid data point, since toxicity
signs may appear regardless of the cross-section angle.

### 8.10 Population Level Image Difference and Acquisition Bias Warning

The average image per class (Section 5.6) revealed a striking
population-level difference between the two classes that warrants
careful interpretation. The retinal structure in the average Si image
appears more diffuse and spread out, while the average No image shows
a more concentrated and well-defined retinal band. The difference map
shows intensity differences that extend beyond the outer retinal layers
where HCQ toxicity is biologically expected to manifest.

This pattern is unlikely to reflect a pure biological signal of
hydroxychloroquine toxicity. Toxicity typically manifests as localized
disruption of specific retinal layers, particularly the photoreceptor
ellipsoid zone, the outer nuclear layer, and the retinal pigment
epithelium in the parafoveal region, which would produce a small,
focused difference signature rather than the broad structural shift
observed here [5][6]. The difference more likely reflects, at least in
part, systematic differences in how the two classes of images were
acquired or processed, such as different OCT equipment gain settings,
different acquisition periods, or different export configurations
between confirmed-toxicity and control patients. Acquisition
heterogeneity is a well-documented source of bias in OCT datasets, as
different devices, scanning protocols, and signal characteristics
introduce domain shifts that challenge model generalization [7][8].

Deep learning models in medical imaging are known to exploit shortcut
features — spurious correlations and acquisition-level signatures that
correlate with class labels but are not causally related to the target
pathology — which can inflate test accuracy by up to 20% while producing
models that fail to generalize to new clinical settings [9][10]. A
convolutional neural network trained directly on these raw images would
risk learning the acquisition signature rather than the actual biological
signs of toxicity. To mitigate this confounder, the following
preprocessing steps can be applied before training:

1. **Intensity normalization per image**
Contrast Limited Adaptive Histogram Equalization (CLAHE) has demonstrated efficacy in improving deep learning robustness across medical imaging modalities acquired with heterogeneous equipment, by removing global brightness differences while preserving local structural information [11][12].

2. **Cropping the overlay watermarks**
The scan number, scale bar, and compass markers visible in the average images contain text and line elements that the model could exploit as shortcut features.

3. **Discussion with Dr. Rodríguez García**
The structural difference will be raised with the clinical sponsor to determine whether the No and Si scans were acquired under comparable conditions or whether acquisition protocol differences may have contributed to the observed
effect.

Beyond the structural difference, the average images also show that the
macula is consistently positioned slightly below the image center in both
classes, with the characteristic foveal pit visible in the average No
image. The retinal layer band in the average Si image appears flatter
and less defined, which could reflect a real biological effect of
toxicity disrupting layer organization [5]. However, this interpretation
cannot be confirmed until the potential acquisition-level confounder is
addressed.

## 9. References

1.   Goceri, E. (2023). Medical image data augmentation: techniques, comparisons and interpretations. Artificial intelligence review, 56(11), 12561-12605.
2.   Kebaili, A., Lapuyade-Lahorgue, J., & Ruan, S. (2023). Deep learning approaches for data augmentation in medical imaging: a review. Journal of imaging, 9(4), 81.
3.   Shorten, C., & Khoshgoftaar, T. M. (2019). A survey on image data augmentation for deep learning. Journal of big data, 6(1), 1-48.
4.   Bergmann, D., & Stryker, C. (2024, June 12). What is a Variational Autoencoder? | IBM. Www.ibm.com. https://www.ibm.com/think/topics/variational-autoencoder

5. Yusuf, I. H., Sharma, S., Luqmani, R., & Downes, S. M. (2017). Hydroxychloroquine retinopathy. Eye (London, England), 31(6), 828–845. https://doi.org/10.1038/eye.2016.298

6. Yusuf IH, Charbel Issa P and Ahn SJ (2023) Hydroxychloroquine-induced Retinal Toxicity. Front. Pharmacol. 14:1196783. doi: 10.3389/fphar.2023.1196783

7. Abdi, A. S., & Abdulazeez, A. M. (2025). A comprehensive review of
deep learning in OCT image segmentation and classification. Medicine in
Novel Technology and Devices, 28, 100396.
https://doi.org/10.1016/j.medntd.2025.100396

8. Tampu, I. E., Eklund, A., & Haj-Hosseini, N. (2022). Inflation of test accuracy due to data leakage in deep learning-based classification of OCT images. Scientific data, 9(1), 580. https://doi.org/10.1038/s41597-022-01618-6

9. Ong Ly, C., Unnikrishnan, B., Tadic, T., Patel, T., Duhamel, J., Kandel, S., Moayedi, Y., Brudno, M., Hope, A., Ross, H., & McIntosh, C. (2024). Shortcut learning in medical AI hinders generalization: method for estimating AI model generalization without external data. NPJ digital medicine, 7(1), 124. https://doi.org/10.1038/s41746-024-01118-4

10. Müller, S., & Berens, P. (2026). Mitigating shortcut learning
via feature disentanglement in medical imaging: A benchmark study.
arXiv preprint arXiv:2602.18502.

11. Yoshimi, Y., Mine, Y., Ito, S., Takeda, S., Okazaki, S., Nakamoto, T.,
Nagasaki, T., Kakimoto, N., Murayama, T., & Tanimoto, K. (2024). Image
preprocessing with contrast-limited adaptive histogram equalization
improves the segmentation performance of deep learning for the articular
disk of the temporomandibular joint on magnetic resonance images. Oral
Surgery, Oral Medicine, Oral Pathology and Oral Radiology, 138(1),
128–141. https://doi.org/10.1016/j.oooo.2023.01.016

12. Wang, Y., Guo, Y., Wang, Z., Yu, L., Yan, Y., & Gu, Z. (2024). Enhancing semantic segmentation in chest X-ray images through image preprocessing: ps-KDE for pixel-wise substitution by kernel density estimation. PloS one, 19(6), e0299623. https://doi.org/10.1371/journal.pone.0299623